# Heart Disease Classification - Model Training & MLflow Experiment Tracking

## MLOps Assignment - Tasks 2 & 3: Feature Engineering, Model Development & Experiment Tracking

This notebook covers:
1. Feature Engineering & Preprocessing Pipeline
2. Model Training (Logistic Regression, Random Forest, Gradient Boosting)
3. Hyperparameter Tuning
4. Cross-Validation & Evaluation
5. MLflow Experiment Tracking
6. Model Selection & Saving

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os

# Add parent directory to path for importing custom modules
sys.path.insert(0, os.path.abspath('..'))

# Sklearn imports
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
# Note: Using sklearn's GradientBoostingClassifier instead of XGBoost
import mlflow
import mlflow.sklearn
import joblib

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print('All libraries imported successfully!')
print(f'MLflow version: {mlflow.__version__}')

## 1. Load Data

In [ ]:
# Load cleaned data from EDA
df = pd.read_csv('../data/heart_disease_cleaned.csv')
print(f'Dataset loaded: {df.shape}')
df.head()

In [ ]:
# Define features and target
NUMERICAL_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
CATEGORICAL_FEATURES = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']
TARGET = 'target'

# Split features and target
X = df.drop(TARGET, axis=1)
y = df[TARGET]

print(f'Features: {X.shape[1]}')
print(f'Numerical: {NUMERICAL_FEATURES}')
print(f'Categorical: {CATEGORICAL_FEATURES}')
print(f'\nTarget distribution:')
print(y.value_counts())

In [ ]:
# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'\nTraining set target distribution:')
print(y_train.value_counts())

## 2. Feature Engineering & Preprocessing Pipeline

In [ ]:
# Create preprocessing pipeline

# Numerical preprocessing: impute + scale
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing: impute + one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, NUMERICAL_FEATURES),
        ('cat', categorical_pipeline, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough'
)

print('Preprocessing pipeline created!')

In [ ]:
# Fit and transform the data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f'Original features: {X_train.shape[1]}')
print(f'Processed features: {X_train_processed.shape[1]}')

In [ ]:
# Save the preprocessor for later use
os.makedirs('../models', exist_ok=True)
joblib.dump(preprocessor, '../models/preprocessor.pkl')
print('Preprocessor saved to ../models/preprocessor.pkl')

## 3. MLflow Setup

In [ ]:
# Set up MLflow
mlflow.set_tracking_uri('file:../mlruns')
experiment_name = 'heart_disease_classification'

# Create or get experiment
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except mlflow.exceptions.MlflowException:
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)
print(f'MLflow experiment: {experiment_name}')
print(f'Experiment ID: {experiment_id}')
print(f'Tracking URI: {mlflow.get_tracking_uri()}')

## 4. Model Training Functions

In [ ]:
def evaluate_model(model, X_test, y_test):
    """Evaluate model and return metrics dictionary."""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_prob)
    }
    return metrics, y_pred, y_prob


def cross_validate_model(model, X, y, cv=5):
    """Perform cross-validation and return results."""
    scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
    results = {}
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    
    for metric in scoring:
        scores = cross_val_score(model, X, y, cv=skf, scoring=metric)
        results[f'cv_{metric}_mean'] = scores.mean()
        results[f'cv_{metric}_std'] = scores.std()
    
    return results


def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'], ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    return fig


def plot_roc_curve(y_true, y_prob, title='ROC Curve'):
    """Plot ROC curve."""
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.fill_between(fpr, tpr, alpha=0.3)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

print('Evaluation functions defined!')

In [ ]:
def train_and_log_model(model, model_name, X_train, y_train, X_test, y_test, params=None):
    """Train a model and log everything to MLflow."""
    
    with mlflow.start_run(run_name=model_name):
        # Log parameters
        mlflow.log_param('model_type', model_name)
        if params:
            mlflow.log_params(params)
        
        # Train model
        model.fit(X_train, y_train)
        
        # Evaluate on test set
        metrics, y_pred, y_prob = evaluate_model(model, X_test, y_test)
        
        # Log test metrics
        for metric_name, value in metrics.items():
            mlflow.log_metric(metric_name, value)
        
        # Cross-validation
        cv_results = cross_validate_model(model, X_train, y_train)
        for metric_name, value in cv_results.items():
            mlflow.log_metric(metric_name, value)
        
        # Log confusion matrix
        cm_fig = plot_confusion_matrix(y_test, y_pred, f'{model_name} - Confusion Matrix')
        mlflow.log_figure(cm_fig, 'confusion_matrix.png')
        plt.close(cm_fig)
        
        # Log ROC curve
        roc_fig = plot_roc_curve(y_test, y_prob, f'{model_name} - ROC Curve')
        mlflow.log_figure(roc_fig, 'roc_curve.png')
        plt.close(roc_fig)
        
        # Log model
        mlflow.sklearn.log_model(model, 'model')
        
        # Print results
        print(f'\n{model_name} Results:')
        print('=' * 50)
        for metric, value in metrics.items():
            print(f'  {metric}: {value:.4f}')
        print(f'  CV ROC-AUC: {cv_results["cv_roc_auc_mean"]:.4f} (+/- {cv_results["cv_roc_auc_std"]:.4f})')
        
        return model, metrics, cv_results

print('Training function defined!')

## 5. Train Models

In [ ]:
# Store results for comparison
results = {}

### 5.1 Logistic Regression

In [ ]:
# Train Logistic Regression
lr_params = {'C': 1.0, 'penalty': 'l2', 'solver': 'liblinear', 'max_iter': 1000}
lr_model = LogisticRegression(**lr_params, random_state=42)

lr_model, lr_metrics, lr_cv = train_and_log_model(
    lr_model, 'Logistic_Regression',
    X_train_processed, y_train,
    X_test_processed, y_test,
    lr_params
)

results['Logistic_Regression'] = {'metrics': lr_metrics, 'cv': lr_cv, 'model': lr_model}

### 5.2 Random Forest

In [ ]:
# Train Random Forest
rf_params = {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2}
rf_model = RandomForestClassifier(**rf_params, random_state=42, n_jobs=-1)

rf_model, rf_metrics, rf_cv = train_and_log_model(
    rf_model, 'Random_Forest',
    X_train_processed, y_train,
    X_test_processed, y_test,
    rf_params
)

results['Random_Forest'] = {'metrics': rf_metrics, 'cv': rf_cv, 'model': rf_model}

### 5.3 Gradient Boosting

In [ ]:
# Train Gradient Boosting (sklearn's GradientBoostingClassifier)
# Note: Using sklearn's GradientBoostingClassifier instead of XGBoost due to libomp dependency issues
gb_params = {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1, 'subsample': 0.8}
gb_model = GradientBoostingClassifier(**gb_params, random_state=42)

gb_model, gb_metrics, gb_cv = train_and_log_model(
    gb_model, 'Gradient_Boosting',
    X_train_processed, y_train,
    X_test_processed, y_test,
    gb_params
)

results['Gradient_Boosting'] = {'metrics': gb_metrics, 'cv': gb_cv, 'model': gb_model}

## 6. Hyperparameter Tuning (Best Model)

In [ ]:
# Find best model so far
comparison_df = pd.DataFrame({
    name: data['metrics'] for name, data in results.items()
}).T

print('Model Comparison (Test Set):')
print('=' * 80)
print(comparison_df.round(4))

best_model_name = comparison_df['roc_auc'].idxmax()
print(f'\nBest model so far: {best_model_name} (ROC-AUC: {comparison_df.loc[best_model_name, "roc_auc"]:.4f})')

In [ ]:
# Hyperparameter tuning for Random Forest (typically performs well)
print('Performing hyperparameter tuning for Random Forest...')

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_tuned = RandomForestClassifier(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(
    rf_tuned, param_grid, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1
)

grid_search.fit(X_train_processed, y_train)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')

In [ ]:
# Train tuned model and log to MLflow
best_rf = grid_search.best_estimator_

best_rf_model, best_rf_metrics, best_rf_cv = train_and_log_model(
    best_rf, 'Random_Forest_Tuned',
    X_train_processed, y_train,
    X_test_processed, y_test,
    grid_search.best_params_
)

results['Random_Forest_Tuned'] = {'metrics': best_rf_metrics, 'cv': best_rf_cv, 'model': best_rf_model}

## 7. Model Comparison & Selection

In [ ]:
# Final comparison
final_comparison = pd.DataFrame({
    name: data['metrics'] for name, data in results.items()
}).T

print('Final Model Comparison:')
print('=' * 80)
print(final_comparison.round(4))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot of all metrics
ax1 = axes[0]
final_comparison.plot(kind='bar', ax=ax1)
ax1.set_title('Model Performance Comparison')
ax1.set_xlabel('Model')
ax1.set_ylabel('Score')
ax1.legend(loc='lower right')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')
ax1.set_ylim([0.7, 1.0])

# ROC curves comparison
ax2 = axes[1]
colors = plt.cm.Set1(np.linspace(0, 1, len(results)))

for (name, data), color in zip(results.items(), colors):
    model = data['model']
    y_prob = model.predict_proba(X_test_processed)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = data['metrics']['roc_auc']
    ax2.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax2.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curves Comparison')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../screenshots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Select best model
best_model_name = final_comparison['roc_auc'].idxmax()
best_model = results[best_model_name]['model']
best_metrics = results[best_model_name]['metrics']

print(f'\n{"="*60}')
print(f'BEST MODEL: {best_model_name}')
print(f'{"="*60}')
for metric, value in best_metrics.items():
    print(f'  {metric}: {value:.4f}')

## 8. Save Final Model

In [ ]:
# Save the best model
joblib.dump(best_model, '../models/model.pkl')
print(f'Best model saved to ../models/model.pkl')

# Also save with MLflow format
with mlflow.start_run(run_name='Best_Model_Final'):
    mlflow.log_param('model_type', best_model_name)
    mlflow.log_param('final_model', True)
    
    for metric, value in best_metrics.items():
        mlflow.log_metric(metric, value)
    
    mlflow.sklearn.log_model(best_model, 'model', registered_model_name='heart_disease_classifier')
    
print('Model registered in MLflow Model Registry')

In [ ]:
# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    # Get feature names after preprocessing
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
    cat_feature_names = cat_encoder.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
    all_feature_names = NUMERICAL_FEATURES + cat_feature_names
    
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[-15:]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(range(len(indices)), importances[indices], align='center')
    ax.set_yticks(range(len(indices)))
    ax.set_yticklabels([all_feature_names[i] if i < len(all_feature_names) else f'Feature_{i}' for i in indices])
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{best_model_name} - Top 15 Feature Importances')
    plt.tight_layout()
    plt.savefig('../screenshots/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Classification report for best model
y_pred_best = best_model.predict(X_test_processed)

print(f'\n{best_model_name} - Classification Report:')
print('=' * 60)
print(classification_report(y_test, y_pred_best, target_names=['No Disease', 'Disease']))

## 9. Summary

In [ ]:
print('=' * 80)
print('MODEL TRAINING SUMMARY')
print('=' * 80)

print(f'''
1. MODELS TRAINED:
   - Logistic Regression (baseline)
   - Random Forest (with default params)
   - Gradient Boosting (with default params)
   - Random Forest (hyperparameter tuned)

2. BEST MODEL: {best_model_name}
   - Accuracy:  {best_metrics['accuracy']:.4f}
   - Precision: {best_metrics['precision']:.4f}
   - Recall:    {best_metrics['recall']:.4f}
   - F1 Score:  {best_metrics['f1_score']:.4f}
   - ROC-AUC:   {best_metrics['roc_auc']:.4f}

3. ARTIFACTS SAVED:
   - Model: ../models/model.pkl
   - Preprocessor: ../models/preprocessor.pkl
   - MLflow runs: ../mlruns/

4. EXPERIMENT TRACKING:
   - All experiments logged to MLflow
   - Parameters, metrics, and artifacts stored
   - Model registered in MLflow Model Registry

5. NEXT STEPS:
   - Build FastAPI inference service
   - Create Docker container
   - Set up CI/CD pipeline
''')

print('=' * 80)

---

## View MLflow UI

To view all experiments and runs, start the MLflow UI:

```bash
cd /path/to/Mlops_assignment
mlflow ui --backend-store-uri file:./mlruns --port 5000
```

Then open http://localhost:5000 in your browser.